# Trend Precursor Test

问：Trend Start 之前几分钟的 momentum，是否和普通时间不一样？

用已有 `trend_events.csv` + 1 分钟 K 线。不做 ML、不搜参、不是回测。逻辑在 `src/qtrader/experiments/trend_precursor.py`。

因子在 **event bar 之前**结束：`mom_N_leadL = log(close[t-L] / close[t-L-N])`，其中 `t` 是 `bar_open`。


In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "qtrader").is_dir() and (_p / "config").is_dir():
        REPO_ROOT = _p
        break
else:
    raise ModuleNotFoundError("Cannot find the qtrader repo")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from qtrader.experiments.trend_precursor import PrecursorConfig, run_precursor_test

print("cwd =", Path.cwd())


cwd = /Users/zihao/work/quant_dev


## 1. 参数

`random_seed=42`。Control：同 symbol、TOD ±30 分钟、距任何 Trend Start ≥ 30 分钟、每个 event 抽 5 个。


In [6]:
CFG = PrecursorConfig(
    events_path="results/trend_events/trend_events.csv",
    output_dir="results/trend_precursor",
    controls_per_event=5,
    tod_tolerance_minutes=30,
    exclusion_minutes=30,
    random_seed=42,
    n_bootstrap=100,
)
CFG


PrecursorConfig(events_path='results/trend_events/trend_events.csv', output_dir='results/trend_precursor', timeframe='1Min', feed='iex', mom_windows=(1, 3, 5, 10, 20), leads=(1, 3, 5, 10), controls_per_event=5, tod_tolerance_minutes=30, exclusion_minutes=30, event_time_pre=30, random_seed=42, n_bootstrap=100, bar_minutes=1)

## 2. 跑：配样本 → 统计 → bootstrap → 图


In [7]:
result = run_precursor_test(CFG)
print(result.output_dir)
print((result.output_dir / "summary.txt").read_text())
result.stats.loc[result.stats["side"]=="directional", [
    "window","lead","n_trend","n_control","mean_diff","cohens_d",
    "auc","auc_ci_lo","auc_ci_hi","top_decile_lift",
]].sort_values(["window","lead"])


results/trend_precursor
Trend precursor test — descriptive, not a fitted model

1. Sample: 187 trends (90 UP / 97 DOWN), 935 controls, 264 sessions. Baseline P(Trend)=0.167.

2. Highest-AUC directional cells (display only, not selected parameters):
  window=1 lead=5  AUC=0.528  diff=0.00004  top-decile lift=1.17
  window=3 lead=10  AUC=0.488  diff=-0.00005  top-decile lift=0.80
  window=1 lead=10  AUC=0.483  diff=-0.00005  top-decile lift=0.90
  window=20 lead=10  AUC=0.477  diff=-0.00024  top-decile lift=1.01
  window=20 lead=5  AUC=0.462  diff=-0.00039  top-decile lift=0.90

3. Directional AUC range 0.327 … 0.528 (median 0.424). Any bootstrap 95% CI excluding 0.5: True.

4. Median top-decile lift 0.82. Any mean-diff CI excluding 0: True.

5. Sample too small for a stable claim: no.

6. Leakage checks: factor_end_time < bar_open and < trend_start; momentum does not cross a session; controls are ≥ exclusion_window from every Trend Start. This is not a backtest.

Display only — do not t

,window,lead,n_trend,n_control,mean_diff,cohens_d,auc,auc_ci_lo,auc_ci_hi,top_decile_lift
2,1,1,187,908,-0.000153,-0.394545,0.371225,0.320378,0.415319,0.585561
5,1,3,187,908,-0.000119,-0.321962,0.393714,0.342953,0.442198,0.692027
8,1,5,187,895,0.000039,0.102454,0.528300,0.469779,0.577890,1.167836
11,1,10,187,905,-0.000052,-0.135159,0.482716,0.449595,0.526811,0.902479
14,3,1,187,886,-0.000373,-0.591989,0.327362,0.286939,0.369827,0.425035
17,3,3,187,876,-0.000181,-0.286567,0.411801,0.359863,0.466945,0.690639
20,3,5,187,878,-0.000031,-0.050163,0.461136,0.413882,0.507380,1.011295
23,3,10,187,884,-0.000052,-0.079438,0.488041,0.444768,0.535572,0.795455
26,5,1,187,860,-0.000439,-0.551496,0.334641,0.288626,0.384062,0.479908
29,5,3,187,859,-0.000248,-0.313341,0.402725,0.350883,0.456813,0.905628
